In [1]:
# =======================================================
# CELL: INFERENCE 3 MODEL × 3 TASK = 9 KẾT QUẢ
#
# 3 Model (mỗi model best cho 1 tác vụ):
#   MODEL_AT      — train trên dữ liệu không che
#   MODEL_MT1     — train trên dữ liệu che 1 sự kiện
#   MODEL_MT_FULL — train trên dữ liệu che toàn bộ
#
# 3 Task (3 cách chuẩn bị input từ data_230_people.csv):
#   AT      — không che thông tin thời gian
#   MT-1    — che ngẫu nhiên 1 sự kiện/nhân vật
#   MT-full — che toàn bộ thông tin thời gian
#
# Ma trận kết quả:
#              | Task AT | Task MT-1 | Task MT-full
#   Model AT   |   [1]   |    [2]    |     [3]
#   Model MT-1 |   [4]   |    [5]    |     [6]
#   Model MTF  |   [7]   |    [8]    |     [9]
# =======================================================

import torch
import numpy as np
import pandas as pd
import scipy.stats as stats
import re
import os
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers
transformers.logging.set_verbosity_error()
# =======================================================
# 1. CẤU HÌNH — chỉ cần đổi 3 đường dẫn model này
# =======================================================
MODELS = {
    "AT":      "/kaggle/input/models/labrynthlabrynth/model-at/other/default/1",       # ← model train không che
    "MT-1":    "/kaggle/input/models/labrynthlabrynth/model-mt1/other/default/1",      # ← model train che 1
    "MT-full": "/kaggle/input/models/labrynthlabrynth/model-mtfull/other/default/1",   # ← model train che toàn bộ
}

DATA_PATH       = "/kaggle/input/datasets/labrynthlabrynth/230-people/data_230_people.csv"
WORK_DIR        = "/kaggle/working"
BATCH_SIZE      = 128
EVENT_THRESHOLD = 10

os.makedirs(WORK_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}\n")

# =======================================================
# 2. MASK PATTERNS
# =======================================================
date_patterns = [
    r'\b\d{1,2}\s+tháng\s+\d{1,2}\s+năm\s+\d{4}\b',
    r'\b\d{1,2}\s+tháng\s+\d{1,2}\b',
    r'\b\d{1,2}[\.\/\-]\d{1,2}[\.\/\-]\d{2,4}\b',
    r'\bnăm\s+\d{4}\b',
    r'(?<!\d)\b(1[0-9]|20)\d{2}\b(?!\s*(người|km|kg|m|KB|MB|GB))'
]
combined_pattern = "|".join(date_patterns)

def mask_time(text):
    if pd.isna(text):
        return text
    return re.sub(combined_pattern, "[MASK]", str(text), flags=re.IGNORECASE)

# =======================================================
# 3. ĐỌC DATA + TẠO 3 PHIÊN BẢN INPUT
# =======================================================
df = pd.read_csv(DATA_PATH)
df = df.sort_values(by=["Nhân vật", "Năm"]).reset_index(drop=True)
df["Thứ tự"] = df.groupby("Nhân vật").cumcount() + 1

# Task AT: giữ nguyên
df["SK_AT"] = df["Sự kiện"]

# Task MT-full: mask toàn bộ
df["SK_MT_full"] = df["Sự kiện"].apply(mask_time)

# Task MT-1: mask ngẫu nhiên 1 sự kiện/nhân vật
masked_idx = set(
    df.groupby("Nhân vật")
      .apply(lambda g: g.sample(n=1, random_state=42).index[0])
      .values
)
df["SK_MT1"] = df.apply(
    lambda row: mask_time(row["Sự kiện"]) if row.name in masked_idx
                else row["Sự kiện"],
    axis=1
)

TASKS = [
    ("SK_AT",      "AT"),
    ("SK_MT1",     "MT-1"),
    ("SK_MT_full", "MT-full"),
]

print(f"Data: {df['Nhân vật'].nunique()} nhân vật | {len(df)} sự kiện")
print(f"Tasks  : {[t[1] for t in TASKS]}")
print(f"Models : {list(MODELS.keys())}\n")

# =======================================================
# 4. HÀM BUILD PROMPT (nhất quán với lúc train)
# =======================================================
def build_prompt(char_name, event_a, event_b):
    premise    = f"Tiểu sử {char_name}. Sự kiện X: {event_a} | Sự kiện Y: {event_b}"
    hypothesis = "Sự kiện X xảy ra trước Sự kiện Y."
    return premise, hypothesis

# =======================================================
# 5. HÀM INFERENCE 1 NHÂN VẬT
# =======================================================
def infer_timeline(char_name, events, model, tokenizer):
    n = len(events)
    if n < 2:
        ranks = list(range(1, n + 1))
        return 1.0, 1.0, True, ranks, ranks

    rng         = np.random.default_rng(seed=42)
    shuf_idx    = list(range(n))
    rng.shuffle(shuf_idx)
    shuf_events = [events[i] for i in shuf_idx]

    pairs, pair_idx = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            ta, tb = build_prompt(char_name, shuf_events[i], shuf_events[j])
            pairs.append((ta, tb))
            pair_idx.append((i, j))

    borda = np.zeros(n)
    for k in range(0, len(pairs), BATCH_SIZE):
        batch = pairs[k:k + BATCH_SIZE]
        enc   = tokenizer(
            [p[0] for p in batch], [p[1] for p in batch],
            return_tensors="pt", truncation=True,
            max_length=128, padding="max_length"
        )
        with torch.no_grad():
            scores = model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device)
            ).logits.squeeze(-1).cpu().numpy()

        for s_i, score in enumerate(scores):
            i, j = pair_idx[k + s_i]
            borda[i] += score

        del enc, scores
    torch.cuda.empty_cache()

    pred_order  = np.argsort(borda)[::-1]
    pred_mapped = [shuf_idx[i] for i in pred_order]
    truth       = list(range(n))

    pred_ranks = np.zeros(n, dtype=int)
    for rank, orig in enumerate(pred_mapped):
        pred_ranks[orig] = rank + 1
    true_ranks = list(range(1, n + 1))

    tau, _ = stats.kendalltau(truth, pred_mapped)
    if np.isnan(tau):
        tau = sum(1 for a, b in zip(truth, pred_mapped) if a == b) / n

    rho, _ = stats.spearmanr(true_ranks, pred_ranks)
    if np.isnan(rho):
        rho = 1.0

    return tau, rho, (truth == pred_mapped), true_ranks, pred_ranks.tolist()

# =======================================================
# 6. HÀM CHẠY 1 CẶP (model_name × task_name)
# =======================================================
def run_one(df, event_col, task_name, model_name, model, tokenizer):
    rows = []
    all_taus, all_rhos = [], []
    short_taus, long_taus = [], []
    exact_count = 0

    loop = tqdm(
        df.groupby("Nhân vật"),
        desc=f"  Model={model_name:8} | Task={task_name:8}",
        leave=False
    )
    for name, group in loop:
        events     = group[event_col].tolist()
        num_events = len(events)
        if num_events < 3:
            continue

        tau, rho, exact, t_ranks, p_ranks = infer_timeline(
            name, events, model, tokenizer
        )

        all_taus.append(tau)
        all_rhos.append(rho)
        if exact:
            exact_count += 1
        (short_taus if num_events <= EVENT_THRESHOLD else long_taus).append(tau)

        for i in range(num_events):
            rows.append({
                "Model":       model_name,
                "Task":        task_name,
                "Nhân vật":   name,
                "Số sự kiện": num_events,
                "Sự kiện":    events[i],
                "Thực tế":    t_ranks[i],
                "Dự đoán":    p_ranks[i],
                "Đúng":       "✓" if t_ranks[i] == p_ranks[i] else "✗",
                "Kendall Tau": round(tau, 4),
                "Spearman Rho":round(rho, 4),
            })

        loop.set_postfix(Tau=f"{np.mean(all_taus):.4f}")

    n_chars    = len(all_taus)
    mean_tau   = np.mean(all_taus)   if all_taus   else 0.0
    mean_rho   = np.mean(all_rhos)   if all_rhos   else 0.0
    exact_rate = exact_count / n_chars * 100 if n_chars else 0.0
    mean_short = np.mean(short_taus) if short_taus else 0.0
    mean_long  = np.mean(long_taus)  if long_taus  else 0.0
    short_acc  = ((mean_short + 1) / 2) * 100
    long_acc   = ((mean_long  + 1) / 2) * 100
    ratio      = (long_acc / short_acc) if short_acc > 0 else 0.0

    summary = {
        "Model":              model_name,
        "Task":               task_name,
        "N_chars":            n_chars,
        "Kendall_Tau":        round(mean_tau, 4),
        "Spearman_Rho":       round(mean_rho, 4),
        "Exact_Match_%":      round(exact_rate, 2),
        f"Short_Acc_%":       round(short_acc, 2),
        f"Long_Acc_%":        round(long_acc, 2),
        "Ratio_L/S":          round(ratio, 2),
        "n_short":            len(short_taus),
        "n_long":             len(long_taus),
    }

    return pd.DataFrame(rows), summary

# =======================================================
# 7. VÒNG LẶP CHÍNH: 3 MODEL × 3 TASK
# =======================================================
all_summaries  = []
all_detail_dfs = []

for model_name, model_path in MODELS.items():
    print(f"\n{'='*55}")
    print(f"📦 Load model: {model_name}  ({model_path})")
    print(f"{'='*55}")

    tokenizer_m = AutoTokenizer.from_pretrained(model_path)
    model_m     = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=1
    ).to(device)
    model_m.eval()

    for event_col, task_name in TASKS:
        detail_df, summary = run_one(
            df, event_col, task_name,
            model_name, model_m, tokenizer_m
        )
        all_detail_dfs.append(detail_df)
        all_summaries.append(summary)
        print(f"  ✅ [{model_name}] × [{task_name}]  "
              f"Tau={summary['Kendall_Tau']:.4f}  "
              f"Rho={summary['Spearman_Rho']:.4f}  "
              f"Exact={summary['Exact_Match_%']:.1f}%")

    # Giải phóng VRAM trước khi load model tiếp theo
    del model_m, tokenizer_m
    torch.cuda.empty_cache()

# =======================================================
# 8. IN KẾT QUẢ — MA TRẬN 3×3
# =======================================================
summary_df = pd.DataFrame(all_summaries)

print("\n" + "=" * 70)
print("📊 KẾT QUẢ TỔNG HỢP — MA TRẬN 3 MODEL × 3 TASK")
print("=" * 70)

# 8a. Bảng Kendall Tau (pivot)
print("\n▸ Kendall Tau:")
tau_pivot = summary_df.pivot(index="Model", columns="Task", values="Kendall_Tau")
tau_pivot = tau_pivot.reindex(index=list(MODELS.keys()), columns=["AT","MT-1","MT-full"])
print(tau_pivot.to_string())

# 8b. Bảng Spearman Rho (pivot)
print("\n▸ Spearman Rho:")
rho_pivot = summary_df.pivot(index="Model", columns="Task", values="Spearman_Rho")
rho_pivot = rho_pivot.reindex(index=list(MODELS.keys()), columns=["AT","MT-1","MT-full"])
print(rho_pivot.to_string())

# 8c. Bảng Exact Match % (pivot)
print("\n▸ Exact Match (%):")
em_pivot = summary_df.pivot(index="Model", columns="Task", values="Exact_Match_%")
em_pivot = em_pivot.reindex(index=list(MODELS.keys()), columns=["AT","MT-1","MT-full"])
print(em_pivot.to_string())

# 8d. Chi tiết từng ô
print("\n" + "=" * 70)
print("▸ Chi tiết Short/Long Acc:")
print("=" * 70)
for _, row in summary_df.iterrows():
    print(f"  [{row['Model']:8}] × [{row['Task']:8}]  "
          f"Short={row['Short_Acc_%']:5.2f}% (n={row['n_short']:3})  "
          f"Long={row['Long_Acc_%']:5.2f}% (n={row['n_long']:3})  "
          f"Ratio={row['Ratio_L/S']:.2f}")

# 8e. Highlight ô tốt nhất theo từng task
print("\n" + "=" * 70)
print("🏆 Model tốt nhất theo từng task (Kendall Tau):")
print("=" * 70)
for task_name in ["AT", "MT-1", "MT-full"]:
    sub  = summary_df[summary_df["Task"] == task_name]
    best = sub.loc[sub["Kendall_Tau"].idxmax()]
    print(f"  Task {task_name:8} → Best model: {best['Model']:8}  "
          f"Tau={best['Kendall_Tau']:.4f}")

# Ô cao nhất toàn bộ
best_row = summary_df.loc[summary_df["Kendall_Tau"].idxmax()]
print(f"\n  🥇 Tốt nhất tổng: "
      f"Model={best_row['Model']} × Task={best_row['Task']}  "
      f"Tau={best_row['Kendall_Tau']:.4f}")

# =======================================================
# 9. LƯU KẾT QUẢ
# =======================================================
# Bảng tóm tắt
summary_path = os.path.join(WORK_DIR, "inference_summary_3x3.csv")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"\n💾 Bảng tóm tắt (9 ô) → {summary_path}")

# Pivot Tau để dễ copy vào báo cáo
pivot_path = os.path.join(WORK_DIR, "inference_pivot_tau.csv")
tau_pivot.to_csv(pivot_path, encoding="utf-8-sig")
print(f"💾 Pivot Kendall Tau   → {pivot_path}")

# Chi tiết từng ô
all_detail = pd.concat(all_detail_dfs, ignore_index=True)
detail_path = os.path.join(WORK_DIR, "inference_detail_3x3.csv")
all_detail.to_csv(detail_path, index=False, encoding="utf-8-sig")
print(f"💾 Chi tiết 9 ô gộp   → {detail_path}")

# Lưu từng ô riêng
for model_name in MODELS:
    for _, task_name in TASKS:
        sub = all_detail[
            (all_detail["Model"] == model_name) &
            (all_detail["Task"]  == task_name)
        ]
        fname = f"detail_{model_name.replace('-','_')}_{task_name.replace('-','_')}.csv"
        sub.to_csv(os.path.join(WORK_DIR, fname), index=False, encoding="utf-8-sig")
print(f"💾 Chi tiết 9 file riêng → {WORK_DIR}/detail_*.csv")

Device: cuda



/tmp/ipykernel_23/3719377293.py:82: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=42).index[0])


Data: 250 nhân vật | 2462 sự kiện
Tasks  : ['AT', 'MT-1', 'MT-full']
Models : ['AT', 'MT-1', 'MT-full']


📦 Load model: AT  (/kaggle/input/models/labrynthlabrynth/model-at/other/default/1)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Model=AT       | Task=AT      :   0%|          | 0/250 [00:00<?, ?it/s]

...

  ✅ [MT-full] × [AT]  Tau=0.5701  Rho=0.6785  Exact=4.0%


  Model=MT-full  | Task=MT-1    :   0%|          | 0/250 [00:00<?, ?it/s]

  ✅ [MT-full] × [MT-1]  Tau=0.5696  Rho=0.6773  Exact=4.4%


  Model=MT-full  | Task=MT-full :   0%|          | 0/250 [00:00<?, ?it/s]

  ✅ [MT-full] × [MT-full]  Tau=0.5634  Rho=0.6717  Exact=4.4%

📊 KẾT QUẢ TỔNG HỢP — MA TRẬN 3 MODEL × 3 TASK

▸ Kendall Tau:
Task         AT    MT-1  MT-full
Model                           
AT       0.9079  0.7968   0.3919
MT-1     0.8847  0.8358   0.5348
MT-full  0.5701  0.5696   0.5634

▸ Spearman Rho:
Task         AT    MT-1  MT-full
Model                           
AT       0.9577  0.8808   0.5010
MT-1     0.9474  0.9189   0.6485
MT-full  0.6785  0.6773   0.6717

▸ Exact Match (%):
Task       AT  MT-1  MT-full
Model                       
AT       32.4   6.0      1.2
MT-1     21.2   8.4      2.8
MT-full   4.0   4.4      4.4

▸ Chi tiết Short/Long Acc:
  [AT      ] × [AT      ]  Short=96.21% (n=131)  Long=94.50% (n=119)  Ratio=0.98
  [AT      ] × [MT-1    ]  Short=88.51% (n=131)  Long=91.31% (n=119)  Ratio=1.03
  [AT      ] × [MT-full ]  Short=71.15% (n=131)  Long=67.89% (n=119)  Ratio=0.95
  [MT-1    ] × [AT      ]  Short=94.57% (n=131)  Long=93.87% (n=119)  Ratio=0.99
  [MT-1    